<div dir="rtl">

# 🤖 08 - Conversational RAG Pipeline (تجارب مسار الـ RAG التفاعلي)

## ما هو هذا الكراس؟
يقدم هذا الكراس تطبيقاً شاملاً ومنظماً لبناء خط أنابيب **Conversational RAG (Retrieval-Augmented Generation)** مستخدماً استيعاب الصفحات المباشرة من الويب عبر `WebBaseLoader` وتخزين المتجهات دائمياً بـ `Chroma DB` والربط بـ `ChatGroq`.

## المحاور الرئيسية التي يتم تناولها:
1. **استيعاب صفحات الويب**: جلب مقالات الويب المعقدة وتنظيف نصوصها.
2. **تقسيم وتكشيف النصوص**: التقطيع الذكي التكراري بـ `RecursiveCharacterTextSplitter`.
3. **التضمين الرقمي وقاعدة المتجهات**: استخدام `HuggingFaceEmbeddings` ومستودع المتجهات `Chroma`.
4. **بناء وتدفق سلاسل RAG التفاعلية**: دمج المسترجع مع قوالب التوجيه المنظمة ونماذج `Groq` التوليدية.

</div>

### 1️⃣ تحميل البيئة والمكتبات وتحديد رابط الاستيعاب

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

# تحميل متغيرات البيئة
load_dotenv(find_dotenv())
print("✅ تم استيراد مكتبات مسار RAG التفاعلي بنجاح.")

### 2️⃣ سحب مقال الويب وتقسيمه إلى قطع نصية (Ingestion & Chunking)

In [ ]:
# رابط مقال أبحاث الوكلاء الذكيين (Autonomous Agents)
ARTICLE_URL = "https://lilianweng.github.io/posts/2023-06-23-agent/"
print(f"🌐 جاري تحميل المقال المرجعي من: {ARTICLE_URL}")

loader = WebBaseLoader(ARTICLE_URL)
raw_documents = loader.load()

# تقسيم المقال إلى قطع متداخلة للحفاظ على السياق
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(raw_documents)

print(f"✅ تم استيعاب المستندات بنجاح. عدد القطع المستخرجة: {len(chunks)} قطعة.")

### 3️⃣ إنشاء التضمينات وبناء قاعدة بيانات المتجهات (Chroma DB)

In [ ]:
# تهيئة نموذج التضمين المحلي المعتمد
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# إنشاء مستودع المتجهات Chroma في الذاكرة وتغذيتها بالقطع النصية
vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print("✅ تم إنشاء قاعدة بيانات Chroma DB وتجهيز الـ Retriever بنجاح.")

### 4️⃣ تهيئة نموذج التوليد وبناء سلسلة RAG بواسطة LCEL

In [ ]:
# تهيئة نموذج المحادثة
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0.2)

# صياغة التوجيه المحدد لمنع الهلوسة
prompt_system = """
You are an expert AI assistant specializing in Autonomous Agents.
Use the following context to provide clear, concise, and factual answers.
If the context does not contain the information, state clearly that you do not know.

Context:
{context}

Question:
{question}
"""

prompt = ChatPromptTemplate.from_messages([
    ("human", prompt_system)
])

# بناء السلسلة باستخدام عامل الربط |
rag_chain = {"context": retriever, "question": RunnablePassthrough()} | prompt | llm

print("✅ تم تركيب وتثبيت سلسلة RAG بنجاح.")

### 5️⃣ تشغيل واختبار الأسئلة المفاهيمية

In [ ]:
test_queries = [
    "What is Task Decomposition in LLM-powered agents?",
    "What are the key components of an autonomous agent system?",
    "Explain Self-Reflection in ReAct framework."
]

for q in test_queries:
    print(f"\n❓ السؤال: {q}")
    print("=" * 70)
    res = rag_chain.invoke(q)
    print(f"🤖 الإجابة الموثقة:\n{res.content}")

<div dir="rtl">

## 💡 الخلاصة العملية:
- تم النجاح في بناء مسار RAG تفاعلي متكامل يقرأ من الويب مباشرة ويقوم بالفهرسة والتأطير بـ Chroma DB والتوليد عبر Groq.
- يمكن استغلال هذه البنية لبناء خوادم API ومشاريع تفاعلية مستقلة.

</div>